In [16]:
import datetime, pandas as pd, requests, csv, sys, time, os, json
header = {'User-Agent': ''}
### Выгрузка header для запроса
with open('settings/user_agents.json', 'r', encoding='utf-8') as f:
    headers_full = json.load(f)

header_first = str(headers_full['chrome'][0])
header['User-Agent'] = header_first

current_path = sys.path[0]

In [17]:
## Получение дат торгов акции

def get_ticker_dates (moex_df):
    # /iss/history/engines/[engine]/markets/[market]/securities/[security]/dates

    # market = "shares"
    # ticker_in = "SBER"
    # response = requests.get(query, headers = header)
    global header

    errors = []

    result = []

    for i in range (0, len(moex_df)):
        ticker_type = moex_df['SUPERTYPE'][i]
        ticker_in = moex_df['TRADE_CODE'][i]

        if ticker_type in ['Инвестиционные паи','Депозитарные расписки','Акции','Ипотечные сертификаты участия']:
            market = "shares"
        elif ticker_type in ['Облигации','Еврооблигации']:
            market = "bonds"
        else:
            market = "shares"

        query = f'http://iss.moex.com/iss/history/engines/stock/markets/{market}/securities/{ticker_in}/dates.json' #универсальный шаблон

        response = requests.get(query, headers = header)
        status_code = response.status_code

        if status_code == 200:
            data = response.json()
            tmp = []
            try:
                date_from = data['dates']['data'][0][0]
                # print(date_from)
                date_till = data['dates']['data'][0][1]
                # print(date_till)

                tmp.append(ticker_in)
                tmp.append(date_from)
                tmp.append(date_till)

                result.append(tmp)

                time.sleep(10)
            except:
                errors.append(ticker_in)
        else:
            print(status_code, ticker_in) ### сделать нормальную обработку ошибок

    result = pd.DataFrame(result, columns=['ticker','date_from','date_till'])
    
    return result, errors

In [18]:
import data_gathering

all_stocks_ru = data_gathering.moex_tickerlists(current_path)

Общее количество объектов на Мосбирже: 3555
Количество акций и депозитарных расписок: 260


In [19]:
moex_df = all_stocks_ru[all_stocks_ru['TRADE_CODE'] != ''][['SUPERTYPE','TRADE_CODE']]
moex_df.reset_index(drop=True, inplace=True)
len(moex_df)

3101

In [20]:
# moex_df_tmp = moex_df.head(10)
result, errors = get_ticker_dates(moex_df)


In [21]:
result.to_excel(("{}/datasets/ticker_lists/ticker_dates.xlsx").format(current_path))

In [ ]:
#тест кейс
len(result) == len(moex_df)

True

In [24]:
result

,ticker,date_from,date_till
0,RU000A101Z74,2020-07-31,2025-03-24
1,RU000A102DB2,2020-11-20,2025-03-24
2,RU000A102GJ8,2020-12-11,2025-03-24
3,RU000A102LU5,2020-12-28,2025-03-24
4,RU000A104B46,2021-12-27,2025-03-24
...,...,...,...
3096,RU000A0HGNG6,2005-11-10,2025-03-24
3097,RU000A0HGNH4,2005-11-10,2025-03-24
3098,RU000A0JPMD2,2008-02-15,2025-03-24
3099,ARSA,2008-04-07,2025-03-24


In [22]:
errors

[]

In [48]:
# max(result[result['date_till']!='null']['date_till'])
# result[result['date_till']!='null']['date_till']
pd.to_datetime(result['date_till'], errors='coerce').max()

Timestamp('2025-03-24 00:00:00')

In [50]:
pd.to_datetime(result['date_from'], errors='coerce').min()

Timestamp('1997-03-24 00:00:00')

In [51]:
result

,ticker,date_from,date_till
0,RU000A101Z74,2020-07-31,2025-03-24
1,RU000A102DB2,2020-11-20,2025-03-24
2,RU000A102GJ8,2020-12-11,2025-03-24
3,RU000A102LU5,2020-12-28,2025-03-24
4,RU000A104B46,2021-12-27,2025-03-24
...,...,...,...
3096,RU000A0HGNG6,2005-11-10,2025-03-24
3097,RU000A0HGNH4,2005-11-10,2025-03-24
3098,RU000A0JPMD2,2008-02-15,2025-03-24
3099,ARSA,2008-04-07,2025-03-24


In [53]:
ticker_in = "SBER"
ticker_type = "Акции"
end_date_mx = "2011-11-21"
start_date_mx = "2025-03-24"
interval = 24
df_ticker = data_gathering.moex_query(ticker_in, ticker_type, end_date_mx, start_date_mx, interval)
df_ticker

,open,close,high,low,value,volume,begin,end,ticker
0,79.00,75.99,79.37,75.85,2.283014e+10,293803290,2011-11-21 00:00:00,2011-11-21 23:59:59,SBER
1,76.40,76.40,77.76,75.10,2.091095e+10,272599350,2011-11-22 00:00:00,2011-11-22 23:59:59,SBER
2,75.00,76.80,77.44,74.42,2.556061e+10,335403030,2011-11-23 00:00:00,2011-11-23 23:59:59,SBER
3,76.73,73.82,77.29,73.82,1.352856e+10,177337880,2011-11-24 00:00:00,2011-11-24 23:59:59,SBER
4,75.31,78.60,78.75,73.75,2.414558e+10,318867810,2011-11-25 00:00:00,2011-11-25 20:00:17,SBER
...,...,...,...,...,...,...,...,...,...
495,104.06,102.47,104.43,102.40,6.334353e+09,61356730,2013-11-05 00:00:00,2013-11-05 23:59:59,SBER
496,103.00,102.59,103.09,102.26,6.788344e+09,66133090,2013-11-06 00:00:00,2013-11-06 23:59:59,SBER
497,102.31,103.55,103.87,102.10,7.853921e+09,76238840,2013-11-07 00:00:00,2013-11-07 23:59:59,SBER
498,102.41,101.86,102.66,101.57,9.092258e+09,89014360,2013-11-08 00:00:00,2013-11-08 23:59:59,SBER


In [32]:
len(result[result['date_till'] == "2025-03-24"])

2982

In [29]:
result[result['date_till'] != "2025-03-24"]

,ticker,date_from,date_till
94,RU000A104636,None,None
227,US78307AAE38,2002-05-20,2012-03-02
229,US78307ACZ49,2002-05-20,2012-03-02
501,RU000A10B5F0,None,None
522,RU000A0ZZ3S5,None,None
...,...,...,...
2860,RU000A10B1S2,None,None
2862,RU000A10B4M9,None,None
2863,RU000A10B4N7,None,None
2864,RU000A10B4P2,None,None
